# 03. Таблица `user_features.csv`

Ноутбук строит итоговую таблицу признаков пользователей на основе:

- `data/interim/lastfm_time_features.csv`
- `data/processed/user_daily_activity.csv`
- `data/processed/listening_sessions.csv`

## 1. Импорт библиотек

In [9]:
from pathlib import Path

import pandas as pd

## 2. Пути проекта и выходной файл

In [10]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INTERIM_DIR              = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR            = PROJECT_ROOT / "data" / "processed"
TIME_FEATURES_PATH       = INTERIM_DIR / "lastfm_time_features.csv"
USER_DAILY_ACTIVITY_PATH = PROCESSED_DIR / "user_daily_activity.csv"
LISTENING_SESSIONS_PATH  = PROCESSED_DIR / "listening_sessions.csv"
USER_FEATURES_PATH       = PROCESSED_DIR / "user_features.csv"

TIME_FEATURES_PATH, USER_FEATURES_PATH

(PosixPath('/Users/zarmeux/hse/course work 1/course-project-listener-classification/data/interim/lastfm_time_features.csv'),
 PosixPath('/Users/zarmeux/hse/course work 1/course-project-listener-classification/data/processed/user_features.csv'))

## 3. Числовые поля и служебные настройки

In [11]:
DAY_PARTS = [
    "Утро",
    "День",
    "Вечер",
    "Ночь"
]
FLOAT_COLUMNS = [
    "active_days_share",
    "morning_share",
    "day_share",
    "evening_share",
    "night_share",
    "weekday_share",
    "weekend_share",
    "avg_daily_plays",
    "median_daily_plays",
    "min_daily_plays",
    "max_daily_plays",
    "std_daily_plays",
    "avg_session_length",
    "median_session_length",
    "max_session_length",
    "avg_plays_per_session",
    "max_plays_per_session",
]

## 4. Функции расчёта таблицы user_features.csv

In [12]:
def validate_columns(df: pd.DataFrame, required_columns: list[str], dataframe_name: str):
    missing_columns = sorted(set(required_columns) - set(df.columns))
    assert len(missing_columns) == 0, (
        f"Missing required columns in {dataframe_name}: {missing_columns}"
    )


def get_main_day_part(user_df: pd.DataFrame) -> str:
    counts = user_df["day_part"].value_counts().to_dict()
    return sorted(DAY_PARTS, key=lambda part: (-counts.get(part, 0), DAY_PARTS.index(part)))[0]


def get_peak_hour(user_df: pd.DataFrame) -> int:
    hour_counts = user_df.groupby("hour").size().reset_index(name="plays")
    hour_counts = hour_counts.sort_values(["plays", "hour"], ascending=[False, True], kind="stable")
    return int(hour_counts.iloc[0]["hour"])


def get_longest_streak(values: list[bool], expected_value: bool) -> int:
    longest = 0
    current = 0

    for value in values:
        if bool(value) is expected_value:
            current += 1
            longest = max(longest, current)
        else:
            current = 0

    return longest


def build_user_features(
    listenings_df: pd.DataFrame,
    user_daily_activity_df: pd.DataFrame,
    listening_sessions_df: pd.DataFrame,
) -> pd.DataFrame:
    validate_columns(
        listenings_df,
        ["username", "datetime", "day_part", "hour", "is_weekend"],
        "listenings_df",
    )
    validate_columns(
        user_daily_activity_df,
        ["username", "date_only", "has_activity", "plays_count"],
        "user_daily_activity_df",
    )
    validate_columns(
        listening_sessions_df,
        ["username", "plays_count", "duration_minutes"],
        "listening_sessions_df",
    )

    listenings_df          = listenings_df.copy()
    user_daily_activity_df = user_daily_activity_df.copy()
    listening_sessions_df  = listening_sessions_df.copy()

    listenings_df["datetime"]              = pd.to_datetime(listenings_df["datetime"])
    listenings_df["is_weekend"]            = listenings_df["is_weekend"].astype(bool)
    user_daily_activity_df["date_only"]    = pd.to_datetime(user_daily_activity_df["date_only"])
    user_daily_activity_df["has_activity"] = user_daily_activity_df["has_activity"].astype(bool)

    feature_rows = []

    for username in sorted(listenings_df["username"].unique()):
        user_listenings = listenings_df.loc[listenings_df["username"] == username].copy()
        user_days = (
            user_daily_activity_df.loc[user_daily_activity_df["username"] == username]
            .sort_values("date_only", kind="stable")
            .copy()
        )
        user_sessions = listening_sessions_df.loc[listening_sessions_df["username"] == username].copy()

        total_plays       = int(len(user_listenings))
        active_days       = int(user_days["has_activity"].sum())
        inactive_days     = int(len(user_days) - active_days)
        active_days_share = active_days / len(user_days) if len(user_days) > 0 else 0.0

        active_day_counts = user_days.loc[user_days["has_activity"], "plays_count"]
        daily_counts = user_days["plays_count"]

        avg_daily_plays    = float(active_day_counts.mean()) if not active_day_counts.empty else 0.0
        median_daily_plays = int(daily_counts.median()) if not daily_counts.empty else 0.0
        min_daily_plays    = int(daily_counts.min()) if not daily_counts.empty else 0
        max_daily_plays    = int(daily_counts.max()) if not daily_counts.empty else 0
        std_daily_plays    = float(daily_counts.std(ddof=0)) if not daily_counts.empty else 0.0

        morning_share = float((user_listenings["day_part"] == "Утро").mean())
        day_share     = float((user_listenings["day_part"] == "День").mean())
        evening_share = float((user_listenings["day_part"] == "Вечер").mean())
        night_share   = float((user_listenings["day_part"] == "Ночь").mean())
        weekday_share = float((~user_listenings["is_weekend"]).mean())
        weekend_share = float(user_listenings["is_weekend"].mean())

        main_day_part      = get_main_day_part(user_listenings)
        peak_hour          = get_peak_hour(user_listenings)
        active_hours_count = int(user_listenings["hour"].nunique())

        session_count         = int(len(user_sessions))
        avg_session_length    = float(user_sessions["duration_minutes"].mean()) if session_count > 0 else 0.0
        median_session_length = int(user_sessions["duration_minutes"].median()) if session_count > 0 else 0
        max_session_length    = int(user_sessions["duration_minutes"].max()) if session_count > 0 else 0
        avg_plays_per_session = float(user_sessions["plays_count"].mean()) if session_count > 0 else 0.0
        max_plays_per_session = int(user_sessions["plays_count"].max()) if session_count > 0 else 0

        activity_flags = user_days["has_activity"].tolist()
        longest_active_streak = get_longest_streak(activity_flags, True)
        longest_inactive_streak = get_longest_streak(activity_flags, False)

        feature_rows.append(
            {
                "username":                username,
                "total_plays":             total_plays,
                "active_days":             active_days,
                "inactive_days":           inactive_days,
                "active_days_share":       active_days_share,
                "avg_daily_plays":         avg_daily_plays,
                "median_daily_plays":      median_daily_plays,
                "min_daily_plays":         min_daily_plays,
                "max_daily_plays":         max_daily_plays,
                "std_daily_plays":         std_daily_plays,
                "morning_share":           morning_share,
                "day_share":               day_share,
                "evening_share":           evening_share,
                "night_share":             night_share,
                "weekday_share":           weekday_share,
                "weekend_share":           weekend_share,
                "main_day_part":           main_day_part,
                "peak_hour":               peak_hour,
                "active_hours_count":      active_hours_count,
                "session_count":           session_count,
                "avg_session_length":      avg_session_length,
                "median_session_length":   median_session_length,
                "max_session_length":      max_session_length,
                "avg_plays_per_session":   avg_plays_per_session,
                "max_plays_per_session":   max_plays_per_session,
                "longest_active_streak":   longest_active_streak,
                "longest_inactive_streak": longest_inactive_streak,
            }
        )

    result = pd.DataFrame(feature_rows).sort_values("username", kind="stable").reset_index(drop=True)
    result[FLOAT_COLUMNS] = result[FLOAT_COLUMNS].round(6)
    return result

## 5. Загрузка входных таблиц

In [13]:
time_features_df       = pd.read_csv(TIME_FEATURES_PATH)
user_daily_activity_df = pd.read_csv(USER_DAILY_ACTIVITY_PATH)
listening_sessions_df  = pd.read_csv(LISTENING_SESSIONS_PATH)

print(f"time_features_df:       {time_features_df.shape}")
print(f"user_daily_activity_df: {user_daily_activity_df.shape}")
print(f"listening_sessions_df:  {listening_sessions_df.shape}")

time_features_df:       (166153, 15)
user_daily_activity_df: (341, 18)
listening_sessions_df:  (1465, 19)


## 6. Построение итоговой таблицы признаков пользователей

In [14]:
user_features_df = build_user_features(
    listenings_df         =time_features_df,
    user_daily_activity_df=user_daily_activity_df,
    listening_sessions_df =listening_sessions_df,
)

print(f"user_features_df: {user_features_df.shape}")

user_features_df: (11, 27)


## 7. Проверка структуры и сохранение результата

In [15]:
assert not user_features_df.empty
assert user_features_df["username"].is_unique
assert (user_features_df["morning_share"]
        + user_features_df["day_share"]
        + user_features_df["evening_share"]
        + user_features_df["night_share"] - 1).abs().max() < 1e-5
assert (user_features_df["weekday_share"] + user_features_df["weekend_share"] - 1).abs().max() < 1e-5

user_features_df.to_csv(USER_FEATURES_PATH, index=False)
print(f"Saved file: {USER_FEATURES_PATH}")

Saved file: /Users/zarmeux/hse/course work 1/course-project-listener-classification/data/processed/user_features.csv


## 8. Предварительный просмотр итоговой таблицы

In [16]:
user_features_df.head()

,username,total_plays,active_days,inactive_days,active_days_share,avg_daily_plays,median_daily_plays,min_daily_plays,max_daily_plays,std_daily_plays,...,peak_hour,active_hours_count,session_count,avg_session_length,median_session_length,max_session_length,avg_plays_per_session,max_plays_per_session,longest_active_streak,longest_inactive_streak
0,Babs_05,33695,31,0,1.000000,1086.935484,136,33,16302,3031.310607,...,18,24,203,100.039409,27,6794,165.985222,28884,31,0
1,Knapster01,27015,30,1,0.967742,900.500000,102,0,11688,2224.858152,...,22,24,176,110.062500,23,6833,153.494318,22571,27,1
2,Orlenay,10123,31,0,1.000000,326.548387,73,2,3667,721.742195,...,18,24,151,94.675497,23,2303,67.039735,5310,31,0
3,eartle,20966,31,0,1.000000,676.322581,129,19,7878,1532.639772,...,18,24,171,113.181287,28,5479,122.608187,15676,31,0
4,franhale,32712,31,0,1.000000,1055.225806,198,4,15177,2836.126376,...,22,24,191,111.094241,19,6778,171.267016,27802,31,0
